# 5j — Duration patterns by group-contact status (CoMix UK)

Spec: `inst/5_group_contacts.md`. Plan: `inst/5a_group_contacts_plan.md`.

Group-contact flag lives in `multiple_contacts_*` columns of
`dt_comix_no_public/part_uk.jdf`. These columns cover only work / school /
other — there is **no home variant**, so the flag is applied to **non-home**
contacts only. Home is treated as universally no-group.

In [ ]:
include("main_utils.jl")
include("data_setup.jl")
include("comix_uk_time_series.jl")
include("vis_utils.jl")
include("mglm_utils.jl")

default_plot_setting()

## 1. Load CoMix UK + multiple_contacts_* columns; classify participants

Date window: 2021-07-01 ≤ date < 2022-04-01 (Jul 2021 – Mar 2022), mirroring 3j.

The 9 base `multiple_contacts_*` columns are
`{child, adult, older_adult} × {work, school, other}` — no `_phys`,
`_duration`, `_precautions`, `_12_17` variants.

In [ ]:
using CSV, DataFrames, JDF; JDF.save("/workdir/dt_comix_no_public/part_uk.jdf",
  CSV.read("/workdir/dt_comix_no_public/part_uk.csv", DataFrame))

In [ ]:
df, df_part = read_comix_uk_contact_raw()
df_part = @subset(df_part, Date(2021, 7, 1) .<= :date .< Date(2022, 4, 1))
df = innerjoin(df, @select(df_part, :part_id_d, :date),
               on = [:part_id_d, :date])

println("# contacts (after Jul21–Mar22 filter): ", nrow(df))
println("# participant-diary-days:             ", nrow(unique(@select(df_part, :part_id_d, :date))))

In [ ]:
_df_part_inspect = safe_jdf_load("../dt_comix_no_public/part_uk.jdf";
    cols = [:part_wave_uid, :hhld_wave_uid,
        :part_age_group, :part_gender_nb, :date, :wave])

In [ ]:
# Reload part_uk.jdf to pull the 9 multiple_contacts_* base columns.
# `part_id_d` in df_part was created by `rename_part_id_to_uid!` from
# `part_wave_uid`, so we join on `part_wave_uid` (cast to String).
_mc_cols = [
    :multiple_contacts_child_work,        :multiple_contacts_child_school,        :multiple_contacts_child_other,
    :multiple_contacts_adult_work,        :multiple_contacts_adult_school,        :multiple_contacts_adult_other,
    :multiple_contacts_older_adult_work,  :multiple_contacts_older_adult_school,  :multiple_contacts_older_adult_other,
]

df_mc = safe_jdf_load("../dt_comix_no_public/part_uk.jdf";
                       cols = vcat([:part_wave_uid, :date], _mc_cols))
for c in _mc_cols
    df_mc[!, c] = [ismissing(v) ? missing : string(v) for v in df_mc[!, c]]
end
df_mc = @subset(df_mc, Date(2021, 7, 1) .<= :date .< Date(2022, 4, 1))
@rename!(df_mc, :part_id_d = :part_wave_uid)
df_mc[!, :part_id_d] = string.(df_mc.part_id_d)

# Keep only the (part_id_d, date) rows that survive our df_part filter.
df_mc = innerjoin(df_mc, @select(df_part, :part_id_d, :date),
                  on = [:part_id_d, :date])

println("# participant-day rows with multiple_contacts_* values: ", nrow(df_mc))

In [ ]:
# Classification of a single string value.
#   :no_group       — NA / missing / "no" / "No" / "0" / "no-one/no other people" / "no contacts"
#   :has_group_num  — parses to a strictly positive integer
#   :has_group_yes  — "yes" / "Yes"
#   :has_group_text — any other non-NA string (free-text place names etc.)
const _NO_GROUP_STRINGS = Set(["no", "No", "0", "no-one/no other people", "no contacts"])

function classify_mc(v)::Symbol
    (ismissing(v) || v == "NA" || v == "") && return :no_group
    v in _NO_GROUP_STRINGS && return :no_group
    v == "yes" || v == "Yes" && return :has_group_yes
    # numeric?
    n = tryparse(Int, v)
    if n !== nothing
        return n > 0 ? :has_group_num : :no_group
    end
    f = tryparse(Float64, v)
    if f !== nothing
        return f > 0 ? :has_group_num : :no_group
    end
    return :has_group_text
end

is_has_group(c::Symbol) = c !== :no_group

# Per-column unique-value table with classification (qualitative review per spec).
function print_mc_value_table(df_mc, col::Symbol; topn::Int = 30)
    vals = df_mc[!, col]
    cnts = countmap(vals)
    rows = sort(collect(cnts); by = x -> -x[2])
    println("== ", col, "  (", length(rows), " distinct values) ==")
    @printf("  %-30s  %-16s  %s\n", "value", "class", "count")
    for (v, c) in first(rows, min(topn, length(rows)))
        @printf("  %-30s  %-16s  %d\n",
                ismissing(v) ? "<missing>" : (v == "" ? "<empty>" : v),
                string(classify_mc(v)), c)
    end
    extra = length(rows) - topn
    extra > 0 && println("  … (", extra, " more distinct values omitted)")
    println()
end

for c in _mc_cols
    print_mc_value_table(df_mc, c; topn = 25)
end

In [ ]:
# Per participant-day, has_group = ANY of the 9 base columns is non-no-group.
df_mc_flag = copy(df_mc)
df_mc_flag[!, :has_group] = map(eachrow(df_mc_flag)) do r
    any(is_has_group(classify_mc(r[c])) for c in _mc_cols)
end

# Build (part_id_d, date) sets for the two groups.
_pd_tuple(df) = collect(zip(df.part_id_d, df.date))
ids_group_pd = Set(_pd_tuple(@subset(df_mc_flag,  :has_group)))
ids_solo_pd  = Set(_pd_tuple(@subset(df_mc_flag, .!:has_group)))

println("# participant-days with group contacts:    ", length(ids_group_pd))
println("# participant-days WITHOUT group contacts: ", length(ids_solo_pd))
println("# total participant-days (covered by mc):  ", length(ids_group_pd) + length(ids_solo_pd))

# Sanity: rows in df_part not covered by df_mc (should be small / zero).
all_pd = Set(_pd_tuple(unique(@select(df_part, :part_id_d, :date))))
uncovered = setdiff(all_pd, union(ids_group_pd, ids_solo_pd))
println("# participant-days in df_part but NOT in df_mc: ", length(uncovered))

In [ ]:
# Subset contact tables for the two groups (non-home stratification).
df_pd = DataFrame(part_id_d = first.(collect(ids_solo_pd)),
                  date      = last.(collect(ids_solo_pd)))
df_solo  = innerjoin(df, df_pd, on = [:part_id_d, :date])

df_pd = DataFrame(part_id_d = first.(collect(ids_group_pd)),
                  date      = last.(collect(ids_group_pd)))
df_group = innerjoin(df, df_pd, on = [:part_id_d, :date])

println("# contacts in solo-only participant-days:  ", nrow(df_solo))
println("# contacts in has-group participant-days:  ", nrow(df_group))

## 2. Proportion of duration categories over degree

Three panels: home (no group split, since group-contacts ≡ non-home), and
non-home split by no-group / has-group.

Uses the `dropna_keep_n` variant from 2j: degree (x) includes NA-duration
contacts, proportions are over the non-NA contacts only.

In [ ]:
function _emp_props_for(df_sub; setting)
    inp = prepare_dm_inputs(df_sub; setting = setting,
                            outcome = :duration_multi,
                            K = 5, dropna_keep_n = true)
    return _emp_props_by_n_with_denom(inp.Y, inp.n, inp.n_obs, 5)
end

p_home    = _props_panel_base(_emp_props_for(df;       setting = "home"),
                              5, category_names_viz[:duration_multi];
                              title = "home (all)")
p_nh_solo  = _props_panel_base(_emp_props_for(df_solo;  setting = "non-home"),
                              5, category_names_viz[:duration_multi];
                              title = "non-home — no group")
p_nh_group = _props_panel_base(_emp_props_for(df_group; setting = "non-home"),
                              5, category_names_viz[:duration_multi];
                              title = "non-home — has group")

plot(p_home, p_nh_solo, p_nh_group; layout = (1, 3), size = (1400, 400))

## 2.5 Proportion of participants with group contacts vs non-home degree

For each non-home degree $n$, the fraction of participant-days that ticked
any of the 9 `multiple_contacts_*` base columns. Participant-days with no
non-home contacts (n=0) are kept — they can still report group contacts.


In [ ]:
# Per (part_id_d, date) non-home degree, joined to has_group flag.
df_nh_deg = combine(groupby(@subset(df, :cnt_home .== "false"),
                            [:part_id_d, :date]),
                    nrow => :n_nh)

df_pd_flag = unique(@select(df_mc_flag, :part_id_d, :date, :has_group))
df_pd_deg  = leftjoin(df_pd_flag, df_nh_deg, on = [:part_id_d, :date])
df_pd_deg[!, :n_nh] = coalesce.(df_pd_deg.n_nh, 0)

df_prop_group_nh = combine(groupby(df_pd_deg, :n_nh)) do sub
    (; n_total = nrow(sub),
       n_group = sum(sub.has_group),
       p_group = sum(sub.has_group) / nrow(sub))
end
sort!(df_prop_group_nh, :n_nh)

println("# participant-days summed across degrees: ", sum(df_prop_group_nh.n_total))
println("head:");  show(stdout, "text/plain", first(df_prop_group_nh, 8));  println()
println("tail:");  show(stdout, "text/plain", last(df_prop_group_nh, 6));   println()

let
    emp     = @subset(df_prop_group_nh, :n_total .> 0)
    max_n   = max(1, maximum(emp.n_nh))
    ms      = _marker_size_log(emp.n_total; scale = 2.0)
    p_gn = plot(xlabel = "non-home degree (n)", ylabel = "P(has group contacts)",
                title  = "Proportion of participant-days reporting group contacts",
                legend = false, ylim = (-0.02, 1.02),
                xscale = :log10,
                xticks = _xticks_for_max(max_n),
                xlim   = (0.9, max_n * 1.1),
                size   = (700, 400))
    # `n_nh = 0` won't sit on log axis: nudge to 0.95 only for display.
    x_disp = ifelse.(emp.n_nh .== 0, 0.95, float.(emp.n_nh))
    scatter!(p_gn, x_disp, emp.p_group;
             ms = ms, msw = 0, alpha = 0.75, color = :steelblue)
    p_gn
end


## 3. Dirichlet–multinomial fits for NA imputation

Two fits, both K=5, `dropna_keep_n=true`, `X = [1, log(n)]`:
- `fit_home` — fit on **all** home contacts (group-status not applicable).
- `fit_nh_solo` — fit on the **no-group-contact subset** of non-home contacts
  (per `inst/5_group_contacts.md` §2 (ii)).

In [ ]:
inp_home    = prepare_dm_inputs(df;      setting = "home",     outcome = :duration_multi, K = 5, dropna_keep_n = true)
inp_nh_solo = prepare_dm_inputs(df_solo; setting = "non-home", outcome = :duration_multi, K = 5, dropna_keep_n = true)

fit_home    = fit_mglm_dm(inp_home.X,    inp_home.Y)
fit_nh_solo = fit_mglm_dm(inp_nh_solo.X, inp_nh_solo.Y)

println("--- fit_home ---");    mglm_dm_show(fit_home)
println("--- fit_nh_solo ---"); mglm_dm_show(fit_nh_solo)

JLD2.@save "../dt_intermediate/5j_dm_home.jld2"    fit_home
JLD2.@save "../dt_intermediate/5j_dm_nh_solo.jld2" fit_nh_solo

## 4. CCDF of **total contact duration per participant-day** (hours)

For each participant-day in a given setting, the sum of imputed per-contact
durations. Three NA-imputation variants — identical mapping at the contact
level to §2's framework, then summed per participant-day:

- **(i)** `NA → <5min` (midpoint 2.5 min).
- **(ii)** `NA → Σ_k p_k(n) · t_mid,k` (single DM-expected midpoint).
- **(iii)** `NA → t_mid,k*` with `k* ~ Categorical(p_k(n))` (Monte-Carlo draw,
  one per NA). Seeded for reproducibility — variant (iii) at the per-participant
  level needs an actual draw, because the *expectation* of variant (iii) would
  collapse onto variant (ii).

DM choice mirrors §3: `fit_home` for home, `fit_nh_solo` for non-home.
Level 5 (>4h) midpoint is capped at 4h, so per-contact $t \le 4$h and per-day
totals are bounded by $4 \cdot n_{\max}$ hours.


In [ ]:
# Midpoints in hours (level 5 = >4h is clamped to 4h per the spec).
const _T_MID_H = (2.5/60, 10/60, 37.5/60, 150/60, 240/60)  # 240 min = 4h

_dur_level(v) = _is_dur_na(v) ? missing : _dur_to_int(v)

# Helper: 1×K DM probability row for degree n (P-by-K β).
function _dm_pk(fit::NamedTuple, n::Integer)
    P = size(fit.β, 1)
    Xn = P == 1 ? reshape([1.0], 1, 1) :
                   reshape([1.0, log(Float64(max(n, 1)))], 1, 2)
    return mglm_dm_proportions(fit.β, Xn)             # 1 × 5
end

# Σ_k p_k(n) · t_mid_h_k.
_dm_expected_midpoint(fit, n) = sum(_dm_pk(fit, n)[1, k] * _T_MID_H[k] for k in 1:5)

# Categorical draw — returns an index in 1..5 with probabilities p_k(n).
function _dm_sample_k(rng::Random.AbstractRNG, fit::NamedTuple, n::Integer)
    p = _dm_pk(fit, n)
    u = rand(rng)
    s = 0.0
    @inbounds for k in 1:5
        s += p[1, k]
        u <= s && return k
    end
    return 5
end


In [ ]:
# Build per-participant totals (in hours) for each variant.
function build_participant_totals(df_in::DataFrame, fit::NamedTuple;
                                  setting::AbstractString,
                                  rng::Random.AbstractRNG)
    sub = setting == "home" ? @subset(df_in, :cnt_home .== "true") :
                              @subset(df_in, :cnt_home .== "false")

    # Per-cell degree n (NA included), joined back onto contacts.
    deg = combine(groupby(sub, [:part_id_d, :date]), nrow => :n)
    sub = leftjoin(sub, deg, on = [:part_id_d, :date])

    # Per-contact imputed duration under the three variants.
    sub[!, :t_i]   = [_T_MID_H[_is_dur_na(v) ? 1 : _dur_to_int(v)]
                      for v in sub.duration_multi]
    sub[!, :t_ii]  = [_is_dur_na(v) ? _dm_expected_midpoint(fit, n) :
                                       _T_MID_H[_dur_to_int(v)]
                      for (v, n) in zip(sub.duration_multi, sub.n)]
    sub[!, :t_iii] = [_is_dur_na(v) ? _T_MID_H[_dm_sample_k(rng, fit, n)] :
                                       _T_MID_H[_dur_to_int(v)]
                      for (v, n) in zip(sub.duration_multi, sub.n)]

    # Sum per (part_id_d, date).
    tot = combine(groupby(sub, [:part_id_d, :date]),
                  :t_i   => sum => :tot_i,
                  :t_ii  => sum => :tot_ii,
                  :t_iii => sum => :tot_iii,
                  nrow   => :n_contacts)
    return tot
end

Random.seed!(1236)
totals_home = build_participant_totals(df, fit_home;
                                       setting = "home",
                                       rng = Random.default_rng())
Random.seed!(1236)
totals_non  = build_participant_totals(df, fit_nh_solo;
                                       setting = "non-home",
                                       rng = Random.default_rng())

println("home    — participant-days: ", nrow(totals_home),
        "    max total (h): ", round(maximum(totals_home.tot_i); digits = 2))
println("non-home — participant-days: ", nrow(totals_non),
        "    max total (h): ", round(maximum(totals_non.tot_i); digits = 2))


In [ ]:
# Finer xticks (hours): pruned to the panel's data range.
const _CCDF_XTICKS_RAW = [(5/60,  "5m"),  (10/60, "10m"), (30/60, "30m"),
                          (1.0,   "1h"),  (2.0,   "2h"),  (4.0,   "4h"),
                          (8.0,   "8h"),  (16.0,  "16h"), (32.0,  "32h"),
                          (64.0,  "64h"), (128.0, "128h")]
function _ccdf_xticks_for(xmax::Real)
    keep = filter(t -> t[1] <= xmax, _CCDF_XTICKS_RAW)
    isempty(keep) && (keep = _CCDF_XTICKS_RAW[1:1])
    return (first.(keep), last.(keep))
end

function plot_total_ccdf_panel(tot::DataFrame; title::AbstractString)
    xmax = max(maximum(tot.tot_i), maximum(tot.tot_ii), maximum(tot.tot_iii))
    p = plot(; xaxis = :log10, xticks = _ccdf_xticks_for(xmax),
             xlim  = (2.5/60, xmax * 1.1),
             ylim  = (1e-5, 1.2),
             legend = :bottomleft, title = title,
             xlabel = "total contact duration per participant-day (hours)",
             ylabel = "CCDF")
    plot_ccdf_continuous!(p, tot.tot_i;   label = "(i) NA→<5min",
                          color = :black,
                          xlabel = "total contact duration per participant-day (hours)")
    plot_ccdf_continuous!(p, tot.tot_ii;  label = "(ii) DM-expected midpoint",
                          color = :orange,
                          xlabel = "total contact duration per participant-day (hours)")
    plot_ccdf_continuous!(p, tot.tot_iii; label = "(iii) DM Monte-Carlo draw",
                          color = :purple,
                          xlabel = "total contact duration per participant-day (hours)")
    return p
end

p_total_home = plot_total_ccdf_panel(totals_home; title = "home")
p_total_non  = plot_total_ccdf_panel(totals_non;  title = "non-home")
plot(p_total_home, p_total_non; layout = (1, 2), size = (1200, 500),
     plot_title = "Total contact duration per participant-day — NA-imputation variants")


## 5. (Bonus) Non-home only — solo vs has-group CCDF overlay

Variant (i) only (NA → <5min) so the comparison is purely about the
underlying duration distribution, not the imputation. Home is omitted
because group status is undefined there.

In [ ]:
function _ccdf_x_i_nonhome(df_in::DataFrame)
    sub = @subset(df_in, :cnt_home .== "false")
    return [_T_MID_H[_is_dur_na(v) ? 1 : _dur_to_int(v)] for v in sub.duration_multi]
end

x_solo  = _ccdf_x_i_nonhome(df_solo)
x_group = _ccdf_x_i_nonhome(df_group)

p = plot(; xaxis = :log10, xlim = (2.0/60, 5.0), ylim = (1e-5, 1.2),
         legend = :bottomleft,
         title = "non-home — solo vs has-group (variant i)")
plot_ccdf_continuous!(p, x_solo;  label = "no group",  color = :blue,
                      xlabel = "duration per contact (hours)")
plot_ccdf_continuous!(p, x_group; label = "has group", color = :red,
                      xlabel = "duration per contact (hours)")
p